# Tesla Deliveries — End-to-End ML Pipeline (2015–2025)
**Week 2: Internship Project**

A complete ML pipeline covering preprocessing, EDA, feature engineering,  
regression modeling, hyperparameter tuning, and time series forecasting.

---
**Dataset:** `tesla_deliveries_dataset_2015_2025.csv` &nbsp;|&nbsp; **Rows:** 2,640 &nbsp;|&nbsp; **Features:** 12 &nbsp;|&nbsp; **Target:** `Estimated_Deliveries`

## Step 1: Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

df = pd.read_csv("/content/tesla_deliveries_dataset_2015_2025.csv")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

**Note on the dataset:** Average delivery per row is ~9,900 units with a wide range (48 to 25,704).  
Prices span $50K–$120K across models. The data covers Jan 2015 to Dec 2025 across 4 regions and 5 models.

## Step 2: Preprocessing

Auditing data quality before any analysis or modeling.

### 2.1 Missing Values & Duplicates

In [ ]:
missing = df.isnull().sum()
print("Missing values:")
print(missing[missing > 0] if missing.any() else "None found.")

dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f"Removed. New shape: {df.shape}")

**Insight:** No missing values or duplicates across all 2,640 rows. The dataset appears  
pre-cleaned — in production settings, delivery data typically requires imputation for  
months where regional reporting is incomplete or delayed.

### 2.2 Date Construction

In [ ]:
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))
df = df.sort_values('Date').reset_index(drop=True)
print("Date range:", df['Date'].min().strftime('%b %Y'), "to", df['Date'].max().strftime('%b %Y'))
df[['Year', 'Month', 'Date']].head()

### 2.3 Outlier Detection (IQR Method)

In [ ]:
target_cols = ['Estimated_Deliveries', 'Avg_Price_USD']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(target_cols):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"{col}: {n_out} outlier(s) | bounds: {lower:,.0f} to {upper:,.0f}")
    axes[i].boxplot(df[col], patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(f"{col}")
    axes[i].set_ylabel(col)
plt.suptitle("Outlier Detection — Boxplots", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** 12 outliers detected in `Estimated_Deliveries` (above the upper IQR bound of 20,338).  
The maximum recorded is 25,704 units — roughly 2.6x the median. These are not errors;  
they reflect genuine demand spikes in high-volume months. `Avg_Price_USD` is outlier-free,  
staying cleanly within the $50K–$120K range. Both columns are retained as-is.

### 2.4 Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
le = LabelEncoder()
for col in ['Region', 'Model', 'Source_Type']:
    df_encoded[col + '_enc'] = le.fit_transform(df[col])
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

### 2.5 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
              'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print("Scaling applied. Post-scaling mean and std:")
df_scaled[scale_cols].agg(['mean','std']).round(3)

**Note:** Scaling is applied for Linear Regression which is sensitive to feature magnitude.  
Random Forest is scale-invariant and receives unscaled data.

## Step 3: Exploratory Data Analysis (EDA)

### 3.1 Monthly Delivery Trend

In [ ]:
monthly_total = df.groupby('Date')['Estimated_Deliveries'].sum()

plt.figure(figsize=(13, 4))
plt.plot(monthly_total.index, monthly_total.values, color='steelblue', linewidth=1.8)
plt.fill_between(monthly_total.index, monthly_total.values, alpha=0.15, color='steelblue')
plt.title("Total Tesla Deliveries — Monthly Trend (2015–2025)")
plt.xlabel("Date")
plt.ylabel("Total Deliveries")
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

peak = monthly_total.idxmax()
trough = monthly_total.idxmin()
print(f"Peak   : {peak.strftime('%b %Y')} — {monthly_total[peak]:,} deliveries")
print(f"Trough : {trough.strftime('%b %Y')} — {monthly_total[trough]:,} deliveries")
print(f"Range  : {monthly_total.max() - monthly_total.min():,} units spread")

**Insight:** Total monthly deliveries oscillate between 157,228 (May 2020) and 239,351 (Dec 2018)  
— a spread of ~82,000 units. The May 2020 trough maps directly to COVID-19 factory shutdowns  
at Fremont. The Dec 2018 peak reflects the Model 3 production ramp completing after Tesla's  
well-documented manufacturing bottlenecks. The raw line here shows actual month-to-month  
volatility — no smoothing applied — which gives a more honest picture of how delivery  
volumes move.

### 3.2 Deliveries by Region

In [ ]:
region_monthly = df.groupby(['Date', 'Region'])['Estimated_Deliveries'].sum().reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
for region, grp in region_monthly.groupby('Region'):
    ax.plot(grp['Date'], grp['Estimated_Deliveries'], label=region, linewidth=1.4, alpha=0.85)

ax.set_title("Deliveries by Region — Raw Monthly")
ax.set_xlabel("Date")
ax.set_ylabel("Deliveries")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.show()

region_totals = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=False)
for r, v in region_totals.items():
    print(f"  {r:<15}: {v:>10,}")

**Insight:** Showing raw (unsmoothed) regional data reveals the actual month-to-month  
noise in each region. Middle East leads with 6.70M cumulative deliveries, narrowly ahead  
of Asia (6.54M), Europe (6.49M), and North America (6.46M). The margin between first  
and last is under 4% — unrealistically balanced for real delivery data, pointing to  
synthetic generation. Region is therefore not a strong differentiating signal for modeling.

### 3.3 Deliveries by Model

In [ ]:
model_totals = df.groupby('Model')['Estimated_Deliveries'].sum().sort_values()

plt.figure(figsize=(8, 4))
bars = plt.barh(model_totals.index, model_totals.values, color='steelblue', alpha=0.75)
plt.bar_label(bars, labels=[f"{v/1e6:.2f}M" for v in model_totals.values], padding=4)
plt.title("Cumulative Deliveries by Model (2015–2025)")
plt.xlabel("Total Deliveries")
plt.tight_layout()
plt.show()

**Insight:** All five models fall within a 5.1M–5.4M band over 10 years — essentially equal.  
In reality, Cybertruck only launched in late 2023 and cannot match a decade of Model S volume.  
This confirms the dataset is synthetically balanced across models, which is a known limitation  
and worth stating clearly if this analysis is presented to a technical audience.

### 3.4 Price vs. Deliveries

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(df['Avg_Price_USD'], df['Estimated_Deliveries'],
            alpha=0.3, color='steelblue', edgecolors='none', s=18)
plt.title("Avg Price vs. Estimated Deliveries")
plt.xlabel("Average Price (USD)")
plt.ylabel("Estimated Deliveries")
plt.tight_layout()
plt.show()

corr = df['Avg_Price_USD'].corr(df['Estimated_Deliveries'])
print(f"Pearson correlation: {corr:.4f}")

**Insight:** Pearson correlation of -0.0275 — essentially zero. Price has no meaningful  
linear relationship with delivery volume in this dataset. The scatter is uniformly diffuse  
across the full $50K–$120K range, confirming price was correctly excluded from the feature set.

### 3.5 Correlation Heatmap

In [ ]:
numeric_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
                'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

plt.figure(figsize=(8, 6))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

**Insight:** `CO2_Saved_tons` is strongly correlated with `Estimated_Deliveries` since it is  
derived from it. `Battery_Capacity_kWh` and `Range_km` are correlated with each other —  
larger batteries enable longer range. Neither pair enters our feature set, so multicollinearity  
is not a modeling concern here, but good to document.

## Step 4: Feature Engineering

We aggregate to monthly totals and build time-series features.  
Importantly, Linear Regression and Random Forest receive different feature sets —  
each model gets what suits it best.

In [ ]:
monthly_df = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()
monthly_df.columns = ['Date', 'Estimated_Deliveries']
monthly_df = monthly_df.sort_values('Date').reset_index(drop=True)

# Lag features
monthly_df['lag_1'] = monthly_df['Estimated_Deliveries'].shift(1)
monthly_df['lag_2'] = monthly_df['Estimated_Deliveries'].shift(2)
monthly_df['lag_3'] = monthly_df['Estimated_Deliveries'].shift(3)

# Rolling mean — only used for Random Forest
monthly_df['rolling_mean_3'] = monthly_df['Estimated_Deliveries'].rolling(window=3).mean()

# Calendar features
monthly_df['month'] = monthly_df['Date'].dt.month
monthly_df['year']  = monthly_df['Date'].dt.year

monthly_df.dropna(inplace=True)
monthly_df.reset_index(drop=True, inplace=True)

print(f"Feature matrix: {monthly_df.shape[0]} months x {monthly_df.shape[1]} columns")
monthly_df.head()

**Why different feature sets per model:**  
`rolling_mean_3` is a linear combination of `lag_1`, `lag_2`, and `lag_3`.  
Including it in Linear Regression causes multicollinearity — inflated, unstable coefficients  
and artificially perfect metrics. Random Forest handles it fine because it selects features  
independently through tree splits, and the importance scores showed it was genuinely the  
top contributor for RF.

- **LR features:** `lag_1`, `lag_2`, `lag_3`, `month`, `year`  
- **RF features:** `lag_1`, `lag_2`, `lag_3`, `rolling_mean_3`, `month`, `year`

In [ ]:
# Shared base features
lr_features = ['lag_1', 'lag_2', 'lag_3', 'month', 'year']
rf_features  = ['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'month', 'year']

y = monthly_df['Estimated_Deliveries']

# Chronological split — no shuffling
split = int(len(monthly_df) * 0.8)
y_train, y_test = y.iloc[:split], y.iloc[split:]

X_train_lr, X_test_lr = monthly_df[lr_features].iloc[:split], monthly_df[lr_features].iloc[split:]
X_train_rf, X_test_rf = monthly_df[rf_features].iloc[:split], monthly_df[rf_features].iloc[split:]

print(f"Train : {len(y_train)} months  ({monthly_df['Date'].iloc[0].strftime('%b %Y')} — {monthly_df['Date'].iloc[split-1].strftime('%b %Y')})")
print(f"Test  : {len(y_test)} months  ({monthly_df['Date'].iloc[split].strftime('%b %Y')} — {monthly_df['Date'].iloc[-1].strftime('%b %Y')})")

## Step 5: Regression Models

### 5.1 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

lr = LinearRegression()
lr.fit(X_train_lr, y_train)
y_pred_lr = lr.predict(X_test_lr)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mape_lr = np.mean(np.abs((y_test - y_pred_lr) / y_test)) * 100

print(f"Linear Regression — MAE: {mae_lr:,.0f} | RMSE: {rmse_lr:,.0f} | MAPE: {mape_lr:.2f}%")

**Insight:** Without `rolling_mean_3`, Linear Regression now returns an honest error — MAE of 9,960, RMSE of 13,290, and MAPE of 4.97%. On a monthly target averaging ~200K deliveries, a 5% error means the model is off by roughly 10,000 units per month. That is a reasonable baseline for a model working from lag features alone. The previous near-zero score was a direct consequence of including `rolling_mean_3`, which is a linear combination of the lags already in the model — removing it forced LR to actually learn rather than echo back a pre-computed average.

### 5.2 Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_rf, y_train)
y_pred_rf = rf.predict(X_test_rf)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

print(f"Random Forest — MAE: {mae_rf:,.0f} | RMSE: {rmse_rf:,.0f} | MAPE: {mape_rf:.2f}%")

**Insight:** Random Forest scores MAE of 6,696, RMSE of 8,205, and MAPE of 3.37% — meaningfully better than Linear Regression across all three metrics. The gap is now real and interpretable: RF reduces MAE by ~3,264 units (33% improvement) and MAPE by 1.6 percentage points over LR. This difference comes from RF's ability to capture non-linear interactions between lag features, which a linear model cannot. Keeping `rolling_mean_3` in the RF feature set is what enables this — it was the top contributor in the previous run and continues to be here.

### 5.3 Actual vs Predicted

In [ ]:
test_dates = monthly_df['Date'].iloc[split:].values

plt.figure(figsize=(13, 4))
plt.plot(test_dates, y_test.values, label='Actual', color='black', linewidth=1.8)
plt.plot(test_dates, y_pred_lr, label='Linear Regression', linestyle='--', color='steelblue', linewidth=1.4)
plt.plot(test_dates, y_pred_rf, label='Random Forest', linestyle='--', color='tomato', linewidth=1.4)
plt.title("Predicted vs Actual — Test Set (Nov 2023 – Dec 2025)")
plt.xlabel("Date")
plt.ylabel("Total Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

### 5.4 Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': rf_features, 'Importance': importances}).sort_values('Importance')

plt.figure(figsize=(7, 4))
bars = plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue', alpha=0.75)
plt.bar_label(bars, labels=[f"{v:.3f}" for v in feat_df['Importance']], padding=3)
plt.title("Feature Importance — Random Forest")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

**Insight:** `rolling_mean_3` and `lag_1` remain the two dominant features, confirming that short-term delivery momentum drives prediction quality more than anything else. `lag_2` and `lag_3` add diminishing value — the further back you look, the less relevant the signal. `month` and `year` contribute the least, meaning there is no strong within-year seasonality or long-run trend at the aggregated monthly level that the lag features don't already capture.

## Step 6: Hyperparameter Tuning (Random Forest)

GridSearchCV with TimeSeriesSplit — each fold only trains on data preceding the validation window,  
preventing any lookahead bias.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    'n_estimators'     : [50, 100, 200],
    'max_depth'        : [None, 5, 10],
    'min_samples_split': [2, 5]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid, cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
grid_search.fit(X_train_rf, y_train)

print("Best parameters :", grid_search.best_params_)
print(f"Best CV RMSE    : {-grid_search.best_score_:,.0f}")

In [ ]:
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test_rf)

mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
mape_tuned = np.mean(np.abs((y_test - y_pred_tuned) / y_test)) * 100

print(f"Tuned RF — MAE: {mae_tuned:,.0f} | RMSE: {rmse_tuned:,.0f} | MAPE: {mape_tuned:.2f}%")

**Insight:** Best parameters found: `max_depth=10`, `min_samples_split=2`, `n_estimators=200`. The tuned model scores MAE 6,856 and MAPE 3.45% — marginally worse than the default RF (MAE 6,696, MAPE 3.37%). This is a common outcome on smooth, low-noise datasets: the default configuration is already well-calibrated, and adding more trees or constraining depth provides no meaningful gain on the test set. The CV RMSE of 13,293 during training is notably higher than the test RMSE of 8,404, which is a healthy sign — the model is not overfitting to the training period and generalizes well to the Nov 2023–Dec 2025 window.

## Step 7: Time Series Forecasting — ARIMA

ARIMA is used for univariate forecasting on the aggregated monthly series.  
Order (p,d,q) is selected by minimizing AIC across a grid of combinations.

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from statsmodels.tsa.arima.model import ARIMA

ts = monthly_df.set_index('Date')['Estimated_Deliveries'].asfreq('MS')

best_aic, best_order = float("inf"), None

for p in range(4):
    for d in [1]:
        for q in range(3):
            try:
                m = ARIMA(ts, order=(p, d, q)).fit()
                if m.aic < best_aic:
                    best_aic, best_order = m.aic, (p, d, q)
            except:
                continue

print(f"Best ARIMA order : {best_order}")
print(f"Best AIC         : {best_aic:.2f}")

In [ ]:
final_arima = ARIMA(ts, order=best_order).fit()
forecast_result = final_arima.get_forecast(steps=6)
forecast_mean = forecast_result.predicted_mean
forecast_ci   = forecast_result.conf_int(alpha=0.20)

print("6-Month Forecast (Jan–Jun 2026):")
print(pd.DataFrame({
    'Forecast'  : forecast_mean.values.astype(int),
    'Lower 80%' : forecast_ci.iloc[:, 0].values.astype(int),
    'Upper 80%' : forecast_ci.iloc[:, 1].values.astype(int)
}, index=forecast_mean.index.strftime('%b %Y')))

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(ts.index, ts.values, color='steelblue', linewidth=1.6, label='Historical')
plt.plot(forecast_mean.index, forecast_mean.values,
         color='tomato', linewidth=1.8, linestyle='--', label='Forecast')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                 color='tomato', alpha=0.15, label='80% Confidence Interval')
plt.axvline(ts.index[-1], color='gray', linestyle=':', linewidth=1.2, label='Forecast start')
plt.title(f"ARIMA Forecast — Jan to Jun 2026")
plt.xlabel("Date")
plt.ylabel("Total Deliveries")
plt.legend()
plt.tight_layout()
plt.show()

**Insight:** ARIMA(1,1,1) was selected with an AIC of 2,900.10. The (1,1,1) order means the model uses one lag of the differenced series and one lag of the forecast error — a parsimonious specification that suits a series with no strong seasonality. The forecast settles quickly: Jan 2026 at 199,646, dropping to ~197,000 by March and holding flat through June. The 80% confidence band spans roughly ±25,000 units around each point forecast, which is tight relative to the historical range of 82,123 units. The convergence warning during fitting is noted — it does not invalidate the forecast, but suggests the series may benefit from a SARIMA specification if seasonal structure exists at a finer granularity than what this aggregated monthly series shows.

## Model Comparison & Key Takeaways

In [ ]:
results = pd.DataFrame({
    'Model'        : ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)', 'ARIMA(1,1,1)'],
    'Features'     : [
        'lag_1/2/3, month, year',
        'lag_1/2/3, rolling_mean_3, month, year',
        'lag_1/2/3, rolling_mean_3, month, year',
        'Univariate monthly series'
    ],
    'MAE'          : ['9,960', '6,696', '6,856', '—'],
    'RMSE'         : ['13,290', '8,205', '8,404', '—'],
    'MAPE'         : ['4.97%', '3.37%', '3.45%', '—'],
    'AIC'          : ['—', '—', '—', '2,900.10'],
    'Verdict'      : [
        'Honest baseline — rolling mean excluded to avoid multicollinearity',
        'Best test performance — retains rolling_mean_3, genuine 33% MAE gain over LR',
        'Confirms defaults are near-optimal — tuning adds no meaningful improvement',
        'Stable 6-month forecast: 197K–200K/month, ±25K confidence band'
    ]
})
results

---

### Key Takeaways

**Preprocessing**  
No missing values or duplicates across 2,640 rows. The 12 delivery outliers in `Estimated_Deliveries`
(upper IQR bound: 20,338; max observed: 25,704) are genuine demand spikes, not errors — retained to
preserve signal for forecasting. `Avg_Price_USD` is completely outlier-free across its $50K–$120K range.

**EDA**  
Raw monthly deliveries range from 157,228 (May 2020, COVID shutdown) to 239,351 (Dec 2018, Model 3 ramp),
a spread of 82,123 units. All four regions sit within a 4% band of each other (Middle East: 6.70M,
North America: 6.46M), and all five models are similarly uniform — a clear sign of synthetic balancing.
Price correlation with deliveries is -0.0275, confirming it carries no predictive value and was
correctly excluded from the feature set.

**Feature Engineering**  
Two deliberate feature sets were used. LR received lag_1/2/3, month, and year — excluding
`rolling_mean_3` which is a linear combination of the lags and causes multicollinearity in linear models.
RF received the full set including `rolling_mean_3`, which was its top contributor. This design
makes the model comparison genuine: each model is given what it needs, not the same features.

**Modeling**  
LR delivers MAE 9,960 and MAPE 4.97% — an honest baseline now that the redundant feature is removed.
RF (default) outperforms with MAE 6,696 and MAPE 3.37%, a 33% reduction in absolute error.
Hyperparameter tuning (best: max_depth=10, n_estimators=200) returned MAE 6,856 — marginally worse
than the default, confirming the out-of-the-box configuration was already near-optimal for this data.
The CV RMSE of 13,293 vs test RMSE of 8,404 shows healthy generalization with no overfitting.

**Forecasting**  
ARIMA(1,1,1) selected via AIC minimization (AIC: 2,900.10). Forecast converges quickly to ~197K/month
from Jan 2026 through Jun 2026, consistent with the historical plateau. The 80% confidence interval
of ±25K is tight relative to the 82K historical range, indicating reasonable short-term confidence.
For a production system, SARIMA or Prophet would be the natural next step to model any seasonal structure
not visible at this aggregated level.

---
*Pipeline: ingestion → cleaning → EDA → feature engineering → regression → tuning → forecasting.*